In [ ]:
# importo las librerias necesarias
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score


In [5]:
df_ent = pd.read_csv("../data/processed/pedidos_modelo.csv")
print(df_ent.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_state', 'customer_city', 'n_items', 'precio_total', 'flete_total', 'n_vendedores', 'peso_total_g', 'categoria', 'estado_vendedor', 'tarde', 'tarde_dia', 'mismo_estado', 'dias_prometidos', 'tramo_plazo', 'q_dias', 'peso_grupo', 'flete_grupo']


In [7]:
# Variables y target 

num_cols = ["dias_prometidos", "peso_total_g", "flete_total"]
cat_cols = ["customer_state", "estado_vendedor", "categoria"]
bin_cols = ["mismo_estado"]

X = df_ent[num_cols + cat_cols + bin_cols].copy()
y = df_ent["tarde_dia"].astype(int)

In [8]:
# Los 22 pesos en 0 son datos faltantes: los paso a NaN para imputarlos dentro del pipeline
X.loc[X["peso_total_g"] == 0, "peso_total_g"] = np.nan
X["mismo_estado"] = X["mismo_estado"].astype(int)

In [9]:
# Split estratificado 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)
print("% tarde train:", round(y_train.mean() * 100, 2), "| test:", round(y_test.mean() * 100, 2))


(77176, 7) (19294, 7)
% tarde train: 6.77 | test: 6.77


In [10]:
# Baseline: siempre predice "a tiempo"
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
pred = dummy.predict(X_test)
proba = dummy.predict_proba(X_test)[:, 1]
print(classification_report(y_test, pred, zero_division=0))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
print("PR-AUC :", round(average_precision_score(y_test, proba), 3))

              precision    recall  f1-score   support

           0       0.93      1.00      0.96     17987
           1       0.00      0.00      0.00      1307

    accuracy                           0.93     19294
   macro avg       0.47      0.50      0.48     19294
weighted avg       0.87      0.93      0.90     19294

ROC-AUC: 0.5
PR-AUC : 0.068
